# Colab Base para el Trabajo Práctico (Entrega 2)

Programa de creación de la entrega 2

In [ ]:
import pandas as pd
import sqlite3

import sklearn as sk
from sklearn import model_selection
from sklearn import ensemble
from sklearn import metrics

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 0. Lectura de datos

In [ ]:
# TODO: Cambiar para que apunte al directorio correcto
DIR = "/content/drive/MyDrive" + "/datos/propiedades"

In [ ]:
engine = sqlite3.connect(f"{DIR}/entrenamiento.db")

df_ent = pd.read_sql("SELECT * FROM entrenamiento", engine, index_col='id')

In [ ]:
# cantidad de filas y columnas
df_ent.shape

In [ ]:
df_ent.columns

## 1. Entender los datos (AID)

## 2. Limpiar y transformar los datos (DM)

## 2.1. Filtrado de datos



In [ ]:
# Selección de datos. Solo a fines demostrativos.
# TODO: CAMBIAR!
df_ent = df_ent.loc[(df_ent["location_1"] == "Santa Fe") & (df_ent["operation_type"] == 'alquiler') & (df_ent["property_type"] == "departamento")]
df_ent.shape

In [ ]:
# La creación de modelos requiere que no haya valores perdidos
# por eso llenamos todo con 0 a lo bestia
# TODO: Solo se puede modificar por otra constante
df_ent = df_ent.fillna(0)

## 2.2. Tratamiento de valores atípicos

## 2.3. Imputación de valores perdidos

In [ ]:
# TODO: Mejorar la estrategia de imputación.
df_ent = df_ent.fillna(0)

## 2.4. Creación de nuevos atributos

In [ ]:
# Generación de nuevas columnas
# TODO: Crear nuevas variables a partir de datos YA EXISTENTES en el dataset (No usar ningún tipo de fuente externa)
# Tampoco usar librerias ajenas a las que se están viendo en la asignatura.

df_ent["feng-shui"] = 1

## 3. Entrenamiento del modelos (AA)- ⛔⛔⛔ NO TOCAR NADA DE ESTA SECCIÓN ⛔⛔⛔

In [ ]:
# La creación de modelos requiere que todo el dataframe sea numérico
# Me quedo con las columnas numéricas solamente
df_ent = df_ent.select_dtypes('number')

X = df_ent[df_ent.columns.drop('price')]
y = df_ent['price']

In [ ]:
X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(X, y, test_size=0.2, random_state=42)

# Definimos el valor de los hiperparámetros a usar por el modelo
n_estimators = 500
max_depth = 50

# Creamos el modelo a entrenar
reg = sk.ensemble.RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, n_jobs=-1, random_state=42)

# Entrenamos el modelo
_ = reg.fit(X_train, y_train)

# Cálculo del error en entrenamiento (train)
y_pred = reg.predict(X_train)
score_train = sk.metrics.root_mean_squared_error(y_train, y_pred)

# Cálculo del error en prueba (test)
y_pred = reg.predict(X_test)
score_test  = sk.metrics.root_mean_squared_error(y_test,  y_pred)

print(f"{n_estimators=} -- {max_depth=} --> {score_train=:.2f} - {score_test=:.2f}")

## 3.1. (Opcional) Validación Cruzada para mejorar el modelo a partir de los datos

Esta técnica permite mejorar el modelo adaptando los hiperparámetros a los datos seleccionados

**NOTA**: Esta técnica puede tardar mucho, se recomienda ir probando pocos valores en paralelo al resto de los analisis.

In [ ]:
# definimos el valor de los hiperparámetros
n_estimators = 500
max_depth = 50

# Creamos el modelo
# No cambiar RandomForestRegressor
reg = sk.ensemble.RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, n_jobs=-1, random_state=42)

scores_train = []
scores_test = []

# Validación cruzada, 10 folds, shuffle antes, semilla aleatoria
kf = sk.model_selection.KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_index, test_index) in enumerate(kf.split(X, y)):
    # Partimos el fold en entrenamiento y prueba...
    X_train, X_test, y_train, y_test = X.iloc[train_index], X.iloc[test_index], y.iloc[train_index], y.iloc[test_index]

    # Entrenamos el modelo en entramiento
    reg.fit(X_train, y_train)

    # Predecimos en train
    y_pred = reg.predict(X_train)

    # Medimos la performance de la predicción en entramiento
    score_train = sk.metrics.root_mean_squared_error(y_train, y_pred)

    # Predecimos en test
    y_pred = reg.predict(X_test)

    # Medimos la performance de la predicción en prueba
    score_test = sk.metrics.root_mean_squared_error(y_test, y_pred)

    print("\t", f"{fold=}, {score_train=} {score_test=}")


## 3.2. Análisis de la importancia de variables en el modelo (opcional)

Una manera visual de entender a que variable el modelo le está prestando mayor atención.

In [ ]:
feat_importances = pd.Series(reg.feature_importances_, index=X.columns)

# gráfico de barras horizontales
feat_importances.nlargest(10).plot(kind='barh');

## 4. Solución para subir Kaggle

In [ ]:
df_ap = pd.read_csv(f"{DIR}/a_predecir.csv", index_col="id")
df_ap.head(2)

In [ ]:
df_ap.shape

In [ ]:
X = df_ent[df_ent.columns.drop('price')]
y = df_ent['price']

# Entrenamos el modelo con todos los datos de entrenamiento.csv, con los mejores hiperparámetros
n_estimators = 500
max_depth = 50
reg = sk.ensemble.RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, n_jobs=-1, random_state=42)

reg.fit(X, y)

## 4.2. Generación del archivo para Kaggle

In [ ]:

df_ap = pd.read_csv(f"{DIR}/a_predecir.csv", index_col="id")
# TODO: Hacer en df_ap la misma limpieza que en df_ent
df_ap = df_ap.fillna(0)
df_ap["feng-shui"] = 1


df_ap = df_ap.select_dtypes('number')

X_ap = df_ap[X.columns]

# Predecimos los precios del dataset a predecir
y_pred_ap = reg.predict(X_ap)
y_pred_ap

In [ ]:
# Lleno el precio de df_ap con las predicciones
df_ap["price"] = y_pred_ap

# Grabo el df_ap en un archivo csv para subir a Kaggle
nombre = "base-v1"
df_ap["price"].to_csv(f"{DIR}/solucion-{nombre}.csv")